# B.2: Multi-IMV with simulated data

Simulate 900 observations with three outcome classes and use `MulticlassIMV` to compare logistic regression predictions with a constant-only model over three stratified folds.

Run the cell below from any working directory. The base `imvpy` installation supplies the required NumPy, pandas, and scikit-learn dependencies; no data download is needed.

In [1]:
# Install once: python -m pip install imvpy
import pandas as pd
from imvpy import MulticlassIMV
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# 1. Simulate three outcome classes.
X, y = make_classification(
    n_samples=900, n_features=3, n_informative=3,
    n_redundant=0, n_classes=3, n_clusters_per_class=1,
    class_sep=0.5, flip_y=0, random_state=42,
)
features = ["x1", "x2", "x3"]
data = pd.DataFrame(X, columns=features).assign(y=y)

# 2. Evaluate with three stratified folds.
multi = MulticlassIMV(
    data=data, outcome_variable="y",
    optional_explanatory_variables=features,
    model_creator=lambda: LogisticRegression(max_iter=2000),
    n_splits=3, random_state=42, stratified=True,
)
# 3. Report pairwise and one-vs-rest IMV.
_, pairwise = multi.k_fold_imv_matrix()
_, one_vs_rest = multi.k_fold_one_vs_all()
print("Pairwise IMV:")
print(pairwise.round(3))
print("One-vs-rest IMV:")
print(pd.Series(one_vs_rest, index=pairwise.index).round(3))

Pairwise IMV:
       0      1      2
0  0.000  0.554  0.561
1  0.554  0.000  0.649
2  0.561  0.649  0.000
One-vs-rest IMV:
0    0.191
1    0.236
2    0.239
dtype: float64


The first output is the symmetric pairwise Multi-IMV matrix, with classes indexing its rows and columns and zeros on the diagonal. The second reports each class against all remaining classes. Both outputs average IMV over held-out folds; each baseline is fitted on training observations.